### U24AI038 Assignment 8 -A
#### 1. Solve the questions asked in midsem. Write programs to solve them.

In [3]:
# 1. You are building a morphological analyzer for nouns in English language using a finite
# state transducer.
# ‘s’ suffix is concatenated to regular nouns to create plural forms. But
# there is an exception to it when the following occurs:
# A noun ending in s, x, z, ch, or sh forms its plural by adding es.
# Design an FST that can produce all singular and plural forms of nouns as stated above.
# Use +SG for singular features,
# +PL for plural features, and +N for noun features. Show
# all states and transition table as well.

from pathlib import Path
path = Path('../assignment-3/brown_nouns.txt')
data = []
with path.open('r', encoding="utf") as f:
    for i,word in enumerate(f):
        if i<10:
            print(word)
        data.append(word.strip())


len(data)
word_dict = set(data)



class fst:
    def __init__(self, start_state, input_symbols, output_symbols, state_transition, output_function, final_state):
        self.start_state = start_state
        self.input_symbols = input_symbols
        self.output_symbols = output_symbols
        self.state_transition = state_transition
        self.output_function = output_function
        self.final_state = final_state

    def correct(self, string:str):
        end = string.find("+")
        base_word = string[:end]

        if string.find('+SG')!=-1 or string.find('+invalid')!=-1:
            return string if base_word in word_dict else "invalid word"
        i_1 = base_word.rfind("ies")
        if i_1 != -1:
            candidate = base_word[:i_1] +'y'
            if candidate in word_dict:
                return candidate + string[end:]
            else:
                return base_word + "+N+SG" if base_word in word_dict else "invalid word"
        i_2 = base_word.rfind('es')
        if i_2 != -1:
            candidate = base_word[:i_2]
            if candidate in word_dict:
                return base_word[:i_2] + string[end:]
            else:
                return base_word + "+N+SG" if base_word in word_dict else "invalid word"
        n = len(base_word)
        if n >1:
            candidate = base_word[:n-1]
            if candidate in word_dict:
                return base_word[:n-1] + string[end:]
            else:
                return base_word + "+N+SG" if base_word in word_dict else "invalid word"
        return "invalid word"

        
    def parse(self, string):
        curr_state = self.start_state
        #Special fst specifically for this problem: Consists of one final state: qend which is only reached when the string is completed
        qend = self.final_state
        analysis = []
        i =0
        while curr_state != qend:
            curr_input = "" if i >= len(string) else string[i]
            i+=1
            new_state = self.state_transition[curr_state][curr_input]
            curr_output = self.output_function[curr_state][curr_input]
            curr_state = new_state
            if curr_output != "":
                analysis.append(curr_output)

        analysed = "".join(analysis)
        final = self.correct(analysed)
        return final 

        
        
start_state = 'qst'
final_state = 'qend'

states = [
    'qst',
    'qc',
    'q_special',
    'qe',
    'qes',
    'q_s_error',
    'q_s',
    'qend'
]


start_state = 'qst'
final_state = 'qend'

states = [
    'qst',
    'qc',
    'q_special',
    'qe',
    'qes',
    'q_s_error',
    'q_s',
    'qend'
]

import string

inputs = list(string.ascii_letters)

for word in data:
    for ch in word.strip():
        if ch not in inputs:
            inputs.append(ch)

inputs.append('')

outputs = list(string.ascii_letters) + [
    '', '+N+SG', '+N+PL', '+invalid'
]

# States
start_state = 'qst'
final_state = 'qend'

states = [
    'qst',
    'qc',
    'q_special',
    'qe',
    'qes',
    'q_s_error',
    'q_s',
    'qend'
]


# qst

qst_transitions = {}
qst_output = {}

for ch in inputs:

    if ch not in ['c', 's', 'x', 'z', '']:
        qst_transitions[ch] = 'qst'
        qst_output[ch] = ch

qst_transitions['c'] = 'qc'
qst_output['c'] = 'c'

qst_transitions['s'] = 'q_s'
qst_output['s'] = 's'

for ch in ['x', 'z']:
    qst_transitions[ch] = 'q_special'
    qst_output[ch] = ch

qst_transitions[''] = 'qend'
qst_output[''] = '+N+SG'


# qc

qc_transitions = {}
qc_output = {}

for ch in inputs:

    if ch == 'h':
        qc_transitions[ch] = 'q_special'
        qc_output[ch] = 'h'

    elif ch == '':
        qc_transitions[ch] = 'qend'
        qc_output[ch] = '+N+SG'

    else:
        qc_transitions[ch] = 'qst'
        qc_output[ch] = ch


# q_special

q_special_transitions = {}
q_special_output = {}

for ch in inputs:

    if ch == 'e':
        q_special_transitions[ch] = 'qe'
        q_special_output[ch] = 'e'

    elif ch == 's':
        q_special_transitions[ch] = 'q_s_error'
        q_special_output[ch] = 's'

    elif ch == '':
        q_special_transitions[ch] = 'qend'
        q_special_output[ch] = '+N+SG'

    else:
        q_special_transitions[ch] = 'qst'
        q_special_output[ch] = ch


# q_s

q_s_transitions = {}
q_s_output = {}

for ch in inputs:

    if ch == '':
        q_s_transitions[ch] = 'qend'
        q_s_output[ch] = '+N+PL'

    elif ch == 'e':
        q_s_transitions[ch] = 'qe'
        q_s_output[ch] = 'e'

    elif ch == 'h':
        q_s_transitions[ch] = 'q_special'
        q_s_output[ch] = 'h'

    else:
        q_s_transitions[ch] = 'qst'
        q_s_output[ch] = ch


# qe

qe_transitions = {}
qe_output = {}

for ch in inputs:

    if ch == 's':
        qe_transitions[ch] = 'qes'
        qe_output[ch] = 's'

    elif ch == '':
        qe_transitions[ch] = 'qend'
        qe_output[ch] = '+N+SG'

    else:
        qe_transitions[ch] = 'qst'
        qe_output[ch] = ch


# qes

qes_transitions = {}
qes_output = {}

for ch in inputs:

    if ch == '':
        qes_transitions[ch] = 'qend'
        qes_output[ch] = '+N+PL'

    else:
        qes_transitions[ch] = 'qst'
        qes_output[ch] = ch


# q_s_error

q_s_error_transitions = {}
q_s_error_output = {}

for ch in inputs:

    if ch == '':
        q_s_error_transitions[ch] = 'qend'
        q_s_error_output[ch] = '+invalid'

    else:
        q_s_error_transitions[ch] = 'qst'
        q_s_error_output[ch] = ch


transitions = {
    'qst': qst_transitions,
    'qc': qc_transitions,
    'q_special': q_special_transitions,
    'qe': qe_transitions,
    'qes': qes_transitions,
    'q_s': q_s_transitions,
    'q_s_error': q_s_error_transitions
}

output_function = {
    'qst': qst_output,
    'qc': qc_output,
    'q_special': q_special_output,
    'qe': qe_output,
    'qes': qes_output,
    'q_s': q_s_output,
    'q_s_error': q_s_error_output
}

my_fst = fst(
    start_state=start_state,
    input_symbols=inputs,
    output_symbols=outputs,
    state_transition=transitions,
    output_function=output_function,
    final_state=final_state
)

my_fst.parse("irregularities")

investigation

primary

election

evidence

irregularities

place

jury

presentments

charge

election



'irregularity+N+PL'

In [7]:
# 2. Consider the following corpus:
# low lower low lower lowest
# low lowest low lowest
# low new
# new new newer newer
# new
# Apply the wordpiece algorithm for 2 merges. What will be tokenization of the word
# newest ? [CO2] [2+2+2=6]



corpus = [
    "low lower low lower lowest",
    "low lowest low lowest",
    "low new",
    "new new newer newer",
    "new"
]

from collections import defaultdict

def word_piece(corpus, num_merges):
    word_freq = defaultdict(int)

    for sentence in corpus:
        for word in sentence.split():
            word_freq[word] += 1

    # Initialize each unique word as characters
    splits = {
        word: [word[0]] + ["##" + ch for ch in word[1:]]
        for word in word_freq
    }

    vocab = set()
    for tokens in splits.values():
        vocab.update(tokens)

    merge_rules = []

    for _ in range(num_merges):
        pair_freq = defaultdict(int)
        token_freq = defaultdict(int)

        # Count weighted token and pair frequencies
        for word, tokens in splits.items():
            freq = word_freq[word]

            for token in tokens:
                token_freq[token] += freq

            for i in range(len(tokens) - 1):
                pair_freq[(tokens[i], tokens[i + 1])] += freq

        best_pair = None
        best_score = -1

        for (a, b), count in pair_freq.items():
            score = count / (token_freq[a] * token_freq[b])

            if score > best_score:
                best_score = score
                best_pair = (a, b)

        if best_pair is None:
            break

        a, b = best_pair

        # Construct the merged token
        merged = a + b[2:] if b.startswith("##") else a + b

        merge_rules.append((a, b, merged))
        vocab.add(merged)

        # Apply the merge to every word
        for word, tokens in splits.items():
            new_tokens = []
            i = 0

            while i < len(tokens):
                if (
                    i + 1 < len(tokens)
                    and tokens[i] == a
                    and tokens[i + 1] == b
                ):
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            splits[word] = new_tokens

    return merge_rules, vocab, splits    

merge_rules, vocab, splits = word_piece(corpus, 2)
print("Merge Rules:", merge_rules)
print("Vocabulary:", vocab)
print("Splits:", splits)

Merge Rules: [('##s', '##t', '##st'), ('l', '##o', 'lo')]
Vocabulary: {'##e', 'l', '##s', '##t', 'n', 'lo', '##st', '##o', '##w', '##r'}
Splits: {'low': ['lo', '##w'], 'lower': ['lo', '##w', '##e', '##r'], 'lowest': ['lo', '##w', '##e', '##st'], 'new': ['n', '##e', '##w'], 'newer': ['n', '##e', '##w', '##e', '##r']}


In [ ]:
# 3. Consider the following sentences in the toy dataset:
# cat dog animal
# cat cat pet
# dog pet animal
# cat dog pet
# Represent each word in terms of a TF-IDF score. Use raw counts for TF and
# unsmoothed IDF while computing TF-IDF. Which two sentences are closest to each
# other? Show all computations. Don’t add any start and end sentence markers. [Hint: Use
# TF-IDF vectors to represent each sentence.] [CO3] [4+2=6
# ]

documents = [
    "cat dog animal",
    "cat cat pet",
    "dog pet animal",
    "cat dog pet"
]

import math
from collections import defaultdict
def compute_tf_idf(documents):
    documents = [doc.split() for doc in documents]
    num_docs = len(documents)
    inverse_doc_freq = defaultdict(int)
    vocab = set()
    for doc in documents:
        for word in doc:
            vocab.add(word)
    
    for word in vocab: 
        for doc in documents:
            if word in doc:
                inverse_doc_freq[word] += 1
    for word in inverse_doc_freq:
        inverse_doc_freq[word] = math.log(num_docs / inverse_doc_freq[word]) if inverse_doc_freq[word] > 0 else 0
    
    tf_idf_vectors = []
    for doc in documents:
        tf_idf_vector = {}
        for word in vocab:
            tf = doc.count(word)
            idf = inverse_doc_freq[word]
            tf_idf_vector[word] = tf * idf
        tf_idf_vectors.append(tf_idf_vector)
    return tf_idf_vectors

tf_idf_vectors = compute_tf_idf(documents)
print("TF-IDF Vectors:")
for i, vector in enumerate(tf_idf_vectors):
    print(f"Document {i+1}: {vector}")

def cosine_similarity(vec1, vec2):
    dot_product = sum(vec1[word] * vec2[word] for word in vec1)
    magnitude1 = math.sqrt(sum(val ** 2 for val in vec1.values()))
    magnitude2 = math.sqrt(sum(val ** 2 for val in vec2.values()))
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)


best =0
best_pair = None
for i in range(len(tf_idf_vectors)):
    for j in range(i + 1, len(tf_idf_vectors)):
        sim = cosine_similarity(tf_idf_vectors[i], tf_idf_vectors[j])
        if sim > best:
            best = sim
            best_pair = (i, j)

print(f"The closest sentences are Document {best_pair[0]+1} and Document {best_pair[1]+1} with cosine similarity of {best:.4f}")

TF-IDF Vectors:
Document 1: {'dog': 0.28768207245178085, 'cat': 0.28768207245178085, 'animal': 0.6931471805599453, 'pet': 0.0}
Document 2: {'dog': 0.0, 'cat': 0.5753641449035617, 'animal': 0.0, 'pet': 0.28768207245178085}
Document 3: {'dog': 0.28768207245178085, 'cat': 0.0, 'animal': 0.6931471805599453, 'pet': 0.28768207245178085}
Document 4: {'dog': 0.28768207245178085, 'cat': 0.28768207245178085, 'animal': 0.0, 'pet': 0.28768207245178085}
The closest sentences are Document 1 and Document 3 with cosine similarity of 0.8719


In [ ]:
# 4. Design a Naive Baiyes classifier to detect whether a mail is spam or not on the following
# data. free, win, and offer columns represents binary features whether this word appears
# in the email or not.
# Email free win offer Class
# E1 1 1 1 Spam
# E2 1 1 0 Spam
# E3 1 0 1 Spam
# E4 0 1 1 Spam
# E5 0 0 1 NotSpam
# E6 0 1 0 NotSpam
# E7 1 0 0 NotSpam
# What will be your prediction for new email E8 with features (0, 0, 0) where the features
# are in the same order as the above dataset? Show the complete computations. [CO4]
# [4+2=6]

def naive_bayes_classifier(data, new_email):
    # Count the number of spam and not spam emails
    spam_count = sum(1 for email in data if email['Class'] == 'Spam')
    not_spam_count = sum(1 for email in data if email['Class'] == 'NotSpam')
    total_count = len(data)

    # Calculate prior probabilities
    p_spam = spam_count / total_count
    p_not_spam = not_spam_count / total_count

    # Calculate likelihoods
    likelihoods = {
        'Spam': [0, 0, 0],
        'NotSpam': [0, 0, 0]
    }

    for email in data:
        for i in range(3):
            if email['Class'] == 'Spam':
                likelihoods['Spam'][i] += email[f'feature_{i+1}']
            else:
                likelihoods['NotSpam'][i] += email[f'feature_{i+1}']

    # Convert counts to probabilities
    for i in range(3):
        likelihoods['Spam'][i] /= spam_count
        likelihoods['NotSpam'][i] /= not_spam_count

    # Calculate posterior probabilities for the new email
    p_new_email_given_spam = p_spam * likelihoods['Spam'][0] ** new_email[0] * (1 - likelihoods['Spam'][0]) ** (1 - new_email[0]) * \
                             likelihoods['Spam'][1] ** new_email[1] * (1 - likelihoods['Spam'][1]) ** (1 - new_email[1]) * \
                             likelihoods['Spam'][2] ** new_email[2] * (1 - likelihoods['Spam'][2]) ** (1 - new_email[2])

    p_new_email_given_not_spam = p_not_spam * likelihoods['NotSpam'][0] ** new_email[0] * (1 - likelihoods['NotSpam'][0]) ** (1 - new_email[0]) * \
                                 likelihoods['NotSpam'][1] ** new_email[1] * (1 - likelihoods['NotSpam'][1]) ** (1 - new_email[1]) * \
                                 likelihoods['NotSpam'][2] ** new_email[2] * (1 - likelihoods['NotSpam'][2]) ** (1 - new_email[2])


    #print likelihoods and probabilities 
    print("Likelihoods for Spam:", likelihoods['Spam'])
    print("Likelihoods for NotSpam:", likelihoods['NotSpam'])
    print("Prior Probability of Spam:", p_spam)
    print("Prior Probability of NotSpam:", p_not_spam)
    print("Posterior Probability of Spam given new email:", p_new_email_given_spam)
    print("Posterior Probability of NotSpam given new email:", p_new_email_given_not_spam)
    # Make prediction
    if p_new_email_given_spam > p_new_email_given_not_spam:
        return 'Spam'
    else:
        return 'NotSpam' 
    

print("Prediction for new email E8 with features (0, 0, 0):", naive_bayes_classifier([
    {'feature_1': 1, 'feature_2': 1, 'feature_3': 1, 'Class': 'Spam'},
    {'feature_1': 1, 'feature_2': 1, 'feature_3': 0, 'Class': 'Spam'},
    {'feature_1': 1, 'feature_2': 0, 'feature_3': 1, 'Class': 'Spam'},
    {'feature_1': 0, 'feature_2': 1, 'feature_3': 1, 'Class': 'Spam'},
    {'feature_1': 0, 'feature_2': 0, 'feature_3': 1, 'Class': 'NotSpam'},
    {'feature_1': 0, 'feature_2': 1, 'feature_3':   0, 'Class': 'NotSpam'},
    {'feature_1': 1, 'feature_2': 0, 'feature_3': 0, 'Class': 'NotSpam'}
], (0, 0, 0)))

Likelihoods for Spam: [0.75, 0.75, 0.75]
Likelihoods for NotSpam: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]
Prior Probability of Spam: 0.5714285714285714
Prior Probability of NotSpam: 0.42857142857142855
Posterior Probability of Spam given new email: 0.008928571428571428
Posterior Probability of NotSpam given new email: 0.12698412698412703
Prediction for new email E8 with features (0, 0, 0): NotSpam


In [13]:
from collections import defaultdict

def kneser_ney_smoothing(sentences, d=0.5):
    bigram_counts = defaultdict(int)
    unigram_counts = defaultdict(int)
    predecessors = defaultdict(set)
    successors = defaultdict(set)

    for sentence in sentences:
        words = sentence.split()

        for word in words:
            unigram_counts[word] += 1

        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i + 1]
            bigram_counts[(w1, w2)] += 1
            predecessors[w2].add(w1)
            successors[w1].add(w2)

    total_bigram_types = len(bigram_counts)

    def pkn(w2, w1):
        c_bigram = bigram_counts[(w1, w2)]
        c_unigram = unigram_counts[w1]

        if c_unigram == 0:
            return 0.0

        p_cont = (
            len(predecessors[w2]) / total_bigram_types
            if total_bigram_types else 0.0
        )

        lambda_w1 = d * len(successors[w1]) / c_unigram

        return (
            max(c_bigram - d, 0) / c_unigram
            + lambda_w1 * p_cont
        )

    return pkn


sentences = [
    "I like NLP",
    "I like AI",
    "You like NLP",
    "You love AI",
    "We love NLP"
]

pkn = kneser_ney_smoothing(sentences, d=0.5)

print("P(AI | love):", pkn("AI", "love"))
print("P(NLP | like):", pkn("NLP", "like"))

P(AI | love): 0.375
P(NLP | like): 0.5833333333333334
